In [ ]:
import catboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, r2_score
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from tqdm import tqdm

sonar = pd.read_csv("C:/Python/Cases/Sonar/Sonar.csv")
le = LabelEncoder()
sonar['Class'] = le.fit_transform( sonar['Class'] )
X, y = sonar.drop('Class', axis=1), sonar['Class']
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)

gbm = CatBoostClassifier(random_state=25, verbose=0)
gbm.fit(X_train, y_train)
y_pred_prob = gbm.predict_proba(X_test)
print(roc_auc_score(y_test, y_pred_prob[:,1]))

n_est = [10, 50, 100, 200]
rate = np.linspace(0.001, 1, 10)
depth = [1,2,3,4]
scores = []
for i in tqdm(range(len(n_est))):
    for r in rate:
        for d in depth:
            gbm = CatBoostClassifier(random_state=25,  verbose=0,
                    n_estimators=n_est[i], max_depth=d, learning_rate=r)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict_proba(X_test)
            scores.append([n_est[i], r, d, roc_auc_score(y_test, y_pred_prob[:,1])])
    
df_scores = pd.DataFrame(scores, columns=['Estimators', 'learning_rate','max_depth' ,'score'])
df_scores.sort_values('score', ascending=False)

#### Housing

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector

housing = pd.read_csv("C:/Python/Datasets/Housing.csv")
X, y = housing.drop('price', axis=1), housing['price']

X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

Using `OneHotEncoder`

ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
ct = make_column_transformer((ohe, make_column_selector(dtype_include=object)), 
                              remainder='passthrough',
                              verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

X_trn_ohe = ct.fit_transform(X_train)
X_tst_ohe = ct.transform(X_test)

n_est = [10, 50, 100, 200]
rate = np.linspace(0.001, 1, 10)
depth = [1,2,3,4]
scores = []
for i in tqdm(range(len(n_est))):
    for r in rate:
        for d in depth:
            gbm = CatBoostRegressor(random_state=25,  verbose=0,
                    n_estimators=n_est[i], max_depth=d, learning_rate=r)
            gbm.fit(X_trn_ohe, y_train)
            y_pred = gbm.predict(X_tst_ohe)
            scores.append([n_est[i], r, d, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['Estimators', 'learning_rate','max_depth' ,'score'])
df_scores.sort_values('score', ascending=False)

Without Using `OneHotEncoder`

Categorical Columns

cat_cols = list(X_train.columns[X_train.dtypes==object])
cat_cols

gbm = CatBoostRegressor(random_state=25,  verbose=0, cat_features=cat_cols)
gbm.fit(X_train, y_train)
y_pred = gbm.predict(X_test)
r2_score(y_test, y_pred)

n_est = [10, 50, 100, 200]
rate = np.linspace(0.001, 1, 10)
depth = [1,2,3,4]
scores = []
for i in tqdm(range(len(n_est))):
    for r in rate:
        for d in depth:
            gbm = CatBoostRegressor(random_state=25,  verbose=0,cat_features=cat_cols,
                    n_estimators=n_est[i], max_depth=d, learning_rate=r)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            scores.append([n_est[i], r, d, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['Estimators', 'learning_rate','max_depth' ,'score'])
df_scores.sort_values('score', ascending=False)

#### HR Data

hr = pd.read_csv("C:/Python/Cases/human-resources-analytics/HR_comma_sep.csv")
X, y = hr.drop('left', axis=1), hr['left']

X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)

cat_cols = list(X_train.columns[X_train.dtypes==object])
cat_cols

n_est = [10, 50, 100, 200]
rate = np.linspace(0.001, 1, 10)
depth = [1,2,3,4]
scores = []
for i in tqdm(range(len(n_est))):
    for r in rate:
        for d in depth:
            gbm = CatBoostClassifier(random_state=25,  verbose=0,cat_features=cat_cols,
                    n_estimators=n_est[i], max_depth=d, learning_rate=r)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            scores.append([n_est[i], r, d, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['Estimators', 'learning_rate','max_depth' ,'score'])
df_scores.sort_values('score', ascending=False)